<a href="https://colab.research.google.com/github/maicongiehl/unijui-programacao-para-ciencia-de-dados/blob/dev/atividades/estudos-independentes/ei-3/EI_III.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Atividade
Desenvolver aplicações capazes de obter dados automaticamente a partir de fontes disponíveis na Web.

## 1.1 Descrição da atividade

O estudante deverá desenvolver uma aplicação utilizando uma API pública ou técnicas introdutórias de WebScraping para obtenção de dados. Os dados coletados deverão ser armazenados em formato estruturado (CSV ou JSON), possibilitando sua utilização em  atividades posteriores.

O vídeo deverá apresentar o funcionamento da aplicação, a origem dos dados utilizados e a estrutura das informações obtidas.

## 1.2 Entrega:
* Código-fonte ou notebook;
* Arquivo contendo os dados obtidos;
* Vídeo de até 3 minutos.

## 1.3 Critérios de avaliação
* Funcionamento da aplicação desenvolvida;
* Organização do código;
* Estruturação adequada dos dados coletados;
* Documentação da solução;
* Capacidade de demonstrar o funcionamento da aplicação e explicar as decisões adotadas durante seu desenvolvimento.

# 2. Desenvolvimento

## 2.1 Inicialização de ambiente
Instala pacotes e cria variáveis que serão usadas ao longo do código

### 2.1.1 Instalação de dependências

In [ ]:
!pip install BeautifulSoup4 requests

### 2.1.2 Cria variáveis que serão usadas ao longo do código

In [ ]:
from bs4 import BeautifulSoup
import json
import os
from pathlib import Path
import requests

data_dir = 'dados'

categorias_dir = f'{data_dir}/categorias'
lista_livros_dir = f'{data_dir}/livros'
lista_livros_paginados_dir = f'{data_dir}/livros-paginados'
lista_livros_categoria_dir = f'{data_dir}/livros-categoria'
livro_dir = f'{data_dir}/livro'

# Adiciona diretórios necessários em uma lista
dirs = [categorias_dir, lista_livros_dir, lista_livros_paginados_dir, lista_livros_categoria_dir, livro_dir]

# Cria cada diretório da lista, caso não exista
for dir in dirs:
  print(f"Criando diretório {dir}")
  os.makedirs(dir, exist_ok=True)

base_url = 'https://books.toscrape.com'

## 2.2 Criação de Funções para Extração

### 2.2.1 Função para contagem de páginas por URL

In [ ]:
def get_total_pages_by_url(url):
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Busca a quantidade de páginas
    paginacao = soup.select_one("ul.pager li.current")

    if paginacao:
        # Pega o último elemento após dividir por espaços
        texto = paginacao.text.strip()
        total_paginas = int(texto.split()[-1])
        return total_paginas

    # Se a paginação não existir, a listagem tem apenas 1 página
    return 1

### 2.2.2 Função para extração de categorias

In [ ]:
# Busca por categorias de livros
# Pega do arquivo local caso exista, ou na API e persiste
def get_categorias():

  # Pega categorias do ambiente caso já exista
  file_path = Path(f'{categorias_dir}/categorias.json')
  if file_path.is_file():
    with open(file_path) as f:
        return json.load(f)

  # Consulta na  caso não exista
  response = requests.get(base_url)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")
  categorias = []

  # Busca link das categorias
  links_categorias = soup.select("div.side_categories ul.nav-list > li > ul > li > a")

  # Mapeia numeração das categorias
  id = 1
  for link in links_categorias:
      id = id + 1
      # Limpa o nome
      nome = link.text.strip()

      # Adiciona todos os elementos em uma lista
      categorias.append({
          "id": id,
          "categoria": nome.lower(),
          "url": base_url + "/" + link["href"]
      })

  # Persiste os dados em um arquivo local
  with open(f'{categorias_dir}/categorias.json', 'w') as f:
    json.dump(categorias, f)

  return categorias

# Função para buscar o URL da categoria de acordo com o nome
# Utiliza a função de busca pelas categorias
def get_categoria_url_by_name(category: str):
  categorias = get_categorias()

  for item in categorias:
    if item['categoria'] == category:
      print(item['url'])

### 2.2.3 Função para extração de lista de livros por URL

In [ ]:
# Busca lista de livros de acordo com URL
# Usado tanto para busca geral como busca por categorias
def get_lista_livros_by_url(url: str):
  lista_livros = []

  response = requests.get(url)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, 'html.parser')

  # Iteração que busca e trata os dados do HTML
  for livro in soup.select("article.product_pod"):
    titulo = livro.h3.a["title"]

    preco_texto = livro.select_one(".price_color").text
    # Remove caracteres especiais e converte para float
    preco = float(preco_texto.replace("Â", "").replace("£", ""))

    estoque = livro.select_one(".availability").text.strip()

    # Conversão das avaliações, inicialmente em estrelas, para numérico
    classes_rating = livro.select_one(".star-rating")["class"]
    avaliacao = [c for c in classes_rating if c != "star-rating"][0].lower()

    avaliacao_nota = None
    if avaliacao == 'one':
      avaliacao_nota = 1
    elif avaliacao == 'two':
      avaliacao_nota = 2
    elif avaliacao == 'three':
      avaliacao_nota = 3
    elif avaliacao == 'four':
      avaliacao_nota = 4
    elif avaliacao == 'five':
      avaliacao_nota = 5

    link_relativo = livro.h3.a["href"]
    link_completo = f"{base_url}/catalogue/{link_relativo}"

    # Adiciona na lista
    lista_livros.append({
        "titulo": titulo,
        "preco_gbp": preco,
        "estoque": estoque,
        "avaliacao": avaliacao_nota,
        "url": link_completo
    })

  return lista_livros

### 2.2.4 Função para extração de livro por URL

In [ ]:
# Busca por livro de acordo com a URL
def get_book_by_url(url: str):
  import re
  response = requests.get(url)
  soup = BeautifulSoup(response.text, 'html.parser')

  # Extrai informações da tabela
  tabela_info = {}
  for linha in soup.find("table", class_="table-striped").find_all("tr"):
      chave = linha.find("th").text.strip()
      valor = linha.find("td").text.strip()
      tabela_info[chave] = valor

  # Extrai descrição
  desc_header = soup.find("div", id="product_description")
  descricao = desc_header.find_next_sibling("p").text if desc_header else ""

  texto_estoque = tabela_info.get("Availability", "")
  disponivel = "In stock" in texto_estoque

  match_qtd = re.search(r"\d+", texto_estoque)
  qtd_estoque = int(match_qtd.group()) if match_qtd else 0

  livro = {
      "upc": tabela_info.get("UPC"),
      "descricao": descricao,
      "disponivel": disponivel,
      "quantidade_em_estoque": qtd_estoque,
  }
  return livro

## 2.3 Menu Interativo de Extração

In [ ]:
# Menu iterativo para execução de extração
while True:
  print("Bem-vindo ao serviço!")
  print(("[1] Extrair todos os livros"))
  print(("[2] Extrair livros paginados"))
  print(("[3] Extrair todos os livros por categoria"))
  print(("[4] Extrair informações detalhadas do livro"))
  print(("[5] Limpar output"))

  opt = int(input("Digite a opção desejada (0 para sair): "))

  # Extração de todos os livros
  if opt == 1:
    quantidade_paginas = get_total_pages_by_url(base_url)
    print(f"Extraindo {quantidade_paginas} páginas...")

    lista_livros = []
    for i in range(quantidade_paginas):
      url = f'{base_url}/catalogue/page-{i+1}.html'
      print(f"Buscando dados em {url}...")
      lista_livros = lista_livros + get_lista_livros_by_url(url)

    with open(f'{lista_livros_dir}/lista_livros.json', 'w') as f:
        json.dump(lista_livros, f)

  # Extração de livros paginada
  if opt == 2:
    print(f"Extraindo livros paginados...")

    quantidade_paginas = get_total_pages_by_url(base_url)
    opt = int(input('Quantas páginas desejas extrair?'))
    if opt <= 0 or opt >= quantidade_paginas:
      print("Quantidade inválida!")
      continue

    lista_livros = []
    for i in range(opt):
      url = f'{base_url}/catalogue/page-{i+1}.html'
      print(f"Buscando dados em {url}...")
      lista_livros = lista_livros + get_lista_livros_by_url(url)

      with open(f'{lista_livros_paginados_dir}/lista_livros_pagina_{i+1}.json', 'w') as f:
          print(f"Escrevendo página de livros em {lista_livros_paginados_dir}/lista_livros_pagina_{i+1}.json")
          json.dump(lista_livros, f)

  # Extração por categoria
  elif opt == 3:
    categorias = get_categorias()
    for item in categorias:
      print(item['categoria'])

    opt = input("Digite a categoria de interesse: ")
    categoria = None
    for item in categorias:
      if opt == item['categoria']:
        categoria = item

    if categoria is None:
      print("Categoria inválida!")
      break

    print(categoria['url'])
    quantidade_paginas = get_total_pages_by_url(categoria['url'])
    print(f"Extraindo {quantidade_paginas} páginas da categoria {categoria['categoria']}...")

    lista_livros_categoria = []
    if quantidade_paginas == 1:
      url = f'{base_url}/catalogue/category/books/{categoria['categoria'].replace(' ', '-')}_{categoria['id']}/index.html'

      print(f"Buscando dados em {url}...")
      lista_livros_categoria = lista_livros_categoria + get_lista_livros_by_url(url)

    else:
      for i in range(quantidade_paginas):
        url = f'{base_url}/catalogue/category/books/{categoria['categoria'].replace(' ', '-')}_{categoria['id']}/page-{i+1}.html'

        print(f"Buscando dados em {url}...")
        lista_livros_categoria = lista_livros_categoria + get_lista_livros_by_url(url)

    with open(f'{lista_livros_categoria_dir}/lista_livros_{categoria['categoria'].replace(' ', '-')}.json', 'w') as f:
        json.dump(lista_livros_categoria, f)

  # Extração de informações detalhadas por título do livro
  elif opt == 4:

    lista_livros = []
    file_path = Path(f'{lista_livros_dir}/lista_livros.json')
    if file_path.is_file():
      with open(file_path) as f:
          lista_livros = json.load(f)
    else:
      print("É necessário primeiro realizar a extração de todos os livros para conseguir buscar por livro! Use a primeira opção.")

    opt = input('Digite o nome do livro que você deseja realizar o download')
    for item in lista_livros:
      if item['titulo'].lower() == opt.lower():
        print(f"Buscando informações em {item.get('url')}...")
        book = get_book_by_url(item.get('url'))
        file_path = f'./{livro_dir}/{book.get('upc')}.json'
        with open(file_path, 'w') as f:
          print(f"Salvando informações detalhadas em {file_path}...")
          json.dump(book, f)

  # Limpeza do terminal
  elif opt == 5:
    from IPython.display import clear_output
    clear_output(wait=True)

  # Encerra a execução
  elif opt == 0:
    break

### 2.4 Exportação dos Dados Coletados
Célula para compactar a pasta `dados` em um arquivo ZIP e baixar.

In [ ]:
import shutil
from google.colab import files

nome_zip = 'dados_coletados'
caminho = 'dados'

# Cria o arquivo ZIP da pasta 'dados'
shutil.make_archive(nome_zip, 'zip', caminho)
print(f"Arquivo {nome_zip}.zip criado com sucesso!")

# Faz o download do arquivo ZIP gerado
files.download(f"{nome_zip}.zip")